# Merge de JSONs do Label Studio

Este notebook combina dois (ou mais) JSONs exportados do Label Studio em um único arquivo pronto para importar em um novo projeto.

**Funcionalidades:**
- Remove duplicatas por texto normalizado
- Unifica labels (ex: `MULTA_FIXA` + `MULTA_PERCENTUAL` → `MULTA`)
- Renumera IDs sequencialmente
- Limpa referências ao projeto antigo
- Gera estatísticas do dataset resultante

## 1. Configuração

Defina os caminhos dos arquivos e as opções de merge.

In [5]:
# ==============================
# CONFIGURAÇÃO - EDITE AQUI
# ==============================

# Arquivos de entrada (adicione quantos quiser)

from pathlib import Path

curdir = Path.cwd()

INPUT_FILES = [
    curdir / "dataset/labeled_data/decicontas.json",
    curdir / "dataset/labeled_data/decicontas_more_ressarcimentos.json",  # segundo arquivo com mais ressarcimentos
]

# Arquivo de saída
OUTPUT_FILE = "merged_labelstudio.json"

# Remover duplicatas por texto?
DEDUPLICATE = True

# Mapa de unificação de labels (vazio = não unificar)
# Comente/descomente conforme necessário
LABEL_MAP = {
    "MULTA_FIXA": "MULTA",
    "MULTA_PERCENTUAL": "MULTA",
    "OBRIGACAO_MULTA": "OBRIGACAO",
}

# Se não quiser unificar nada, use:
# LABEL_MAP = {}

print("Configuração:")
print(f"  Arquivos: {INPUT_FILES}")
print(f"  Saída: {OUTPUT_FILE}")
print(f"  Dedup: {DEDUPLICATE}")
print(f"  Label map: {LABEL_MAP}")

Configuração:
  Arquivos: [PosixPath('/Users/eduardo/Dev/decicontas.br/dataset/labeled_data/decicontas.json'), PosixPath('/Users/eduardo/Dev/decicontas.br/dataset/labeled_data/decicontas_more_ressarcimentos.json')]
  Saída: merged_labelstudio.json
  Dedup: True
  Label map: {'MULTA_FIXA': 'MULTA', 'MULTA_PERCENTUAL': 'MULTA', 'OBRIGACAO_MULTA': 'OBRIGACAO'}


## 2. Funções auxiliares

In [6]:
import json
import re
from pathlib import Path
from collections import Counter


def normalize_text(text: str) -> str:
    """Normaliza texto para detecção de duplicatas."""
    return re.sub(r'\s+', ' ', text.strip().lower())


def unify_labels(result: list, label_map: dict) -> list:
    """Renomeia labels segundo um mapa (ex: MULTA_FIXA → MULTA)."""
    if not label_map:
        return result
    for r in result:
        if 'value' in r and 'labels' in r['value']:
            r['value']['labels'] = [
                label_map.get(l, l) for l in r['value']['labels']
            ]
    return result


def load_and_normalize(filepath: str) -> list:
    """Carrega JSON do Label Studio (completo ou mínimo)."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{filepath}: esperado lista JSON, encontrado {type(data)}")

    tasks = []
    for item in data:
        # Formato mínimo: {"text": "..."}
        if 'data' not in item and 'text' in item:
            item = {'data': {'text': item['text']}, 'annotations': []}
        if 'annotations' not in item:
            item['annotations'] = []
        tasks.append(item)

    return tasks


def get_label_stats(tasks: list) -> dict:
    """Retorna estatísticas de labels."""
    labels = Counter()
    annotated = 0
    for t in tasks:
        has_ann = False
        for a in t.get('annotations', []):
            for r in a.get('result', []):
                for l in r.get('value', {}).get('labels', []):
                    labels[l] += 1
                    has_ann = True
        if has_ann:
            annotated += 1
    return {'labels': dict(labels.most_common()), 'annotated': annotated, 'total': len(tasks)}


print("Funções carregadas.")

Funções carregadas.


## 3. Explorar os arquivos de entrada

Antes de fazer o merge, vamos ver o que tem em cada arquivo.

In [8]:
for filepath in INPUT_FILES:
    p = Path(filepath)
    if not p.exists():
        print(f"⚠️  {filepath} NÃO ENCONTRADO - verifique o caminho")
        continue

    tasks = load_and_normalize(filepath)
    stats = get_label_stats(tasks)

    print(f"📄 {filepath}")
    print(f"   Tasks: {stats['total']}")
    print(f"   Anotados: {stats['annotated']}")
    print(f"   Labels: {stats['labels']}")
    print()

📄 /Users/eduardo/Dev/decicontas.br/dataset/labeled_data/decicontas.json
   Tasks: 1337
   Anotados: 178
   Labels: {'MULTA_FIXA': 118, 'OBRIGACAO_MULTA': 65, 'RECOMENDACAO': 56, 'OBRIGACAO': 51, 'MULTA_PERCENTUAL': 3, 'RESSARCIMENTO': 2}

📄 /Users/eduardo/Dev/decicontas.br/dataset/labeled_data/decicontas_more_ressarcimentos.json
   Tasks: 88
   Anotados: 57
   Labels: {'MULTA': 83, 'RESSARCIMENTO': 61, 'OBRIGACAO': 4, 'RECOMENDACAO': 2}



## 4. Merge

In [9]:
all_tasks = []
seen_texts = set()
dup_count = 0

for filepath in INPUT_FILES:
    p = Path(filepath)
    if not p.exists():
        print(f"⚠️  Pulando {filepath} (não encontrado)")
        continue

    tasks = load_and_normalize(filepath)
    added = 0

    for task in tasks:
        text = task.get('data', {}).get('text', '')
        norm = normalize_text(text)

        # Dedup
        if DEDUPLICATE and norm in seen_texts:
            dup_count += 1
            continue
        seen_texts.add(norm)

        # Unificar labels
        for ann in task.get('annotations', []):
            ann['result'] = unify_labels(ann.get('result', []), LABEL_MAP)

        all_tasks.append(task)
        added += 1

    print(f"✅ {filepath}: {len(tasks)} lidos → {added} adicionados")

print(f"\n🔄 Duplicatas removidas: {dup_count}")
print(f"📊 Total de tasks no merge: {len(all_tasks)}")

✅ /Users/eduardo/Dev/decicontas.br/dataset/labeled_data/decicontas.json: 1337 lidos → 779 adicionados
✅ /Users/eduardo/Dev/decicontas.br/dataset/labeled_data/decicontas_more_ressarcimentos.json: 88 lidos → 87 adicionados

🔄 Duplicatas removidas: 559
📊 Total de tasks no merge: 866


## 5. Limpar e renumerar IDs

Remove referências ao projeto antigo e atribui IDs sequenciais para evitar conflitos na importação.

In [10]:
fields_to_remove_task = ['project', 'file_upload', 'updated_by', 'comment_authors']
fields_to_remove_ann = ['project', 'task', 'completed_by', 'updated_by']

for i, task in enumerate(all_tasks, start=1):
    task['id'] = i
    task['inner_id'] = i

    for field in fields_to_remove_task:
        task.pop(field, None)

    for ann in task.get('annotations', []):
        for field in fields_to_remove_ann:
            ann.pop(field, None)

print(f"✅ IDs renumerados: 1 → {len(all_tasks)}")

✅ IDs renumerados: 1 → 866


## 6. Estatísticas do dataset final

In [11]:
stats = get_label_stats(all_tasks)

print(f"{'='*50}")
print(f"DATASET FINAL")
print(f"{'='*50}")
print(f"Total de tasks:    {stats['total']}")
print(f"Tasks anotados:    {stats['annotated']}")
print(f"Tasks sem anotação:{stats['total'] - stats['annotated']}")
print(f"{'='*50}")
print(f"Distribuição de labels:")
for label, count in stats['labels'].items():
    bar = '█' * (count // 2)
    print(f"  {label:<20s} {count:>4d}  {bar}")
print(f"{'='*50}")

DATASET FINAL
Total de tasks:    866
Tasks anotados:    229
Tasks sem anotação:637
Distribuição de labels:
  MULTA                 202  █████████████████████████████████████████████████████████████████████████████████████████████████████
  OBRIGACAO             119  ███████████████████████████████████████████████████████████
  RESSARCIMENTO          62  ███████████████████████████████
  RECOMENDACAO           56  ████████████████████████████


## 7. Salvar arquivo para importação

In [12]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_tasks, f, ensure_ascii=False, indent=2)

size_mb = Path(OUTPUT_FILE).stat().st_size / (1024 * 1024)
print(f"💾 Salvo: {OUTPUT_FILE} ({size_mb:.1f} MB)")
print(f"\n📥 Para importar no Label Studio:")
print(f"   1. Crie um novo projeto")
print(f"   2. Vá em Settings > Labeling Interface")
print(f"   3. Configure o template de NER com as labels:")
for label in stats['labels']:
    print(f"      - {label}")
print(f"   4. Vá na aba Import e faça upload de '{OUTPUT_FILE}'")
print(f"   5. As anotações existentes serão preservadas.")

💾 Salvo: merged_labelstudio.json (1.9 MB)

📥 Para importar no Label Studio:
   1. Crie um novo projeto
   2. Vá em Settings > Labeling Interface
   3. Configure o template de NER com as labels:
      - MULTA
      - OBRIGACAO
      - RESSARCIMENTO
      - RECOMENDACAO
   4. Vá na aba Import e faça upload de 'merged_labelstudio.json'
   5. As anotações existentes serão preservadas.


## 8. (Opcional) Amostra de verificação

Visualize algumas tasks anotadas para conferir que o merge está correto.

In [ ]:
# Mostra até 3 tasks anotados como amostra
sample_count = 0
for task in all_tasks:
    if not task['annotations'] or not task['annotations'][0].get('result'):
        continue

    print(f"\n{'─'*60}")
    print(f"Task ID: {task['id']}")
    print(f"Texto (primeiros 200 chars): {task['data']['text'][:200]}...")
    print(f"Labels:")
    for r in task['annotations'][0]['result']:
        labels = r['value'].get('labels', [])
        text_span = r['value'].get('text', '')[:80]
        print(f"  [{', '.join(labels)}] \"{text_span}...\"")

    sample_count += 1
    if sample_count >= 3:
        break

print(f"\n{'─'*60}")
print(f"Mostrando {sample_count} de {stats['annotated']} tasks anotados.")